# Polars Exercises 09 — Lazy Evaluation & Performance

This is Polars' superpower for big data. Instead of loading a whole file into memory (`read_csv`) and then filtering, we describe the *whole query* lazily with `scan_csv`, and Polars builds an optimized plan — reading only the columns and rows we actually need. This is exactly how we keep Foundry transforms fast on millions of rows.

*זה הכוח-על של Polars לנתונים גדולים. במקום לטעון קובץ שלם לזיכרון (`read_csv`) ואז לסנן, אנחנו מתארים את כל השאילתה בעצלות (lazy) עם `scan_csv`, ו-Polars בונה תוכנית מיטבית — קורא רק את העמודות והשורות שבאמת צריך.*

**Data file:** `data/larger.csv` (about 1.1 million rows!)

---

> **גרסת פתרונות** — נסו קודם לבד מתוך `Exercises.ipynb`.

### 1. Import `polars` and the `time` module. Read `data/larger.csv` **eagerly** with `read_csv`, and show its `shape` and `estimated_size("mb")`.

*ייבאו את `polars` ואת `time`. קראו את `data/larger.csv` באופן רגיל (eager) והציגו את ה-`shape` ואת `estimated_size("mb")`.*

In [1]:
import polars as pl
import time

df = pl.read_csv("data/larger.csv")
df.shape, round(df.estimated_size("mb"), 1)

((1100000, 10), 94.2)

### 2. Now create a **LazyFrame** with `scan_csv`. Print its type — note that no data has been read yet; it is just a query plan.

*עכשיו צרו LazyFrame עם `scan_csv`. הדפיסו את הטיפוס שלו — שימו לב שעדיין לא נקרא שום מידע; זו רק תוכנית שאילתה.*

In [2]:
lf = pl.scan_csv("data/larger.csv")
type(lf)

polars.lazyframe.frame.LazyFrame

### 3. Build a lazy query: from the scan, **filter** `category == "pens"` and **select** `cust_num, qty, invoice_total` — but do **not** collect yet. Print the optimized plan with `.explain()`.

*בנו שאילתה עצלה: סננו `category == "pens"` ובחרו 3 עמודות — אבל אל תאספו (collect) עדיין. הדפיסו את התוכנית עם `.explain()`.*

In [3]:
q = (
    pl.scan_csv("data/larger.csv")
    .filter(pl.col("category") == "pens")
    .select("cust_num", "qty", "invoice_total")
)
print(q.explain())

simple π 3/4 ["cust_num", "qty", ... 1 other column]
  Csv SCAN [data/larger.csv]
  PROJECT 4/10 COLUMNS
  SELECTION: [(col("category")) == (String(pens))]


### 4. Look at the plan above: Polars pushed the filter into the scan (`SELECTION`) and reads only 3 of 10 columns (`PROJECT 3/10`). Now run the query with `.collect()` and show the result shape.

*הסתכלו בתוכנית: Polars דחף את הסינון לתוך הקריאה וקורא רק 3 מתוך 10 עמודות. הריצו עכשיו עם `.collect()` והציגו את צורת התוצאה.*

In [4]:
q = (
    pl.scan_csv("data/larger.csv")
    .filter(pl.col("category") == "pens")
    .select("cust_num", "qty", "invoice_total")
)
q.collect().shape

(440142, 3)

### 5. Lazily compute the **total `invoice_total` per `category`**, sorted from highest to lowest. Remember to `collect()`.

*חשבו בעצלות את סך ה-`invoice_total` לכל `category`, ממוין מהגבוה לנמוך. זכרו `collect()`.*

In [5]:
(
    pl.scan_csv("data/larger.csv")
    .group_by("category")
    .agg(pl.col("invoice_total").sum().alias("total"))
    .sort("total", descending=True)
    .collect()
)

category,total
str,f64
"""books""",5.2478e7
"""pens""",2.0976e7
"""pencils""",1.3964e7


### 6. Lazily count how many invoices there are **per `category`**.

*ספרו בעצלות כמה חשבוניות יש לכל `category`.*

In [6]:
(
    pl.scan_csv("data/larger.csv")
    .group_by("category")
    .agg(pl.len().alias("n_invoices"))
    .collect()
)

category,n_invoices
str,u32
"""pens""",440142
"""pencils""",440182
"""books""",219676


### 7. Lazily find the **top 10 `sku`** by total `qty` sold.

*מצאו בעצלות את 10 ה-`sku` המובילים לפי סך ה-`qty` שנמכר.*

In [7]:
(
    pl.scan_csv("data/larger.csv")
    .group_by("sku")
    .agg(pl.col("qty").sum().alias("total_qty"))
    .sort("total_qty", descending=True)
    .head(10)
    .collect()
)

sku,total_qty
str,i64
"""PE1""",2758181
"""PE11""",2747942
"""PE10""",2739411
"""PB24""",2068079
"""PB21""",2065469
"""PB200""",2061844
"""PG22""",2056627
"""BOOK44""",1042204
"""BOOK1""",1029812


### 8. Lazily compute the **average `discount_rate` per `category`**, rounded to 3 decimals.

*חשבו בעצלות את `discount_rate` הממוצע לכל `category`, מעוגל ל-3 ספרות.*

In [8]:
(
    pl.scan_csv("data/larger.csv")
    .group_by("category")
    .agg(pl.col("discount_rate").mean().round(3).alias("avg_discount"))
    .collect()
)

category,avg_discount
str,f64
"""pencils""",0.153
"""pens""",0.153
"""books""",0.153


### 9. Lazily count rows where `qty > 10` **and** `category` is in `["pens", "books"]`.

*ספרו בעצלות שורות שבהן `qty > 10` וגם `category` ב-`["pens", "books"]`.*

In [9]:
(
    pl.scan_csv("data/larger.csv")
    .filter((pl.col("qty") > 10) & (pl.col("category").is_in(["pens", "books"])))
    .select(pl.len())
    .collect()
)

len
u32
348668


### 10. **Profile** the per-category revenue query with `.profile()`. It returns `(result, timings)` — show the `timings` table to see how long each step took.

*הריצו `.profile()` על שאילתת ההכנסה לפי קטגוריה. היא מחזירה `(result, timings)` — הציגו את טבלת ה-`timings`.*

In [10]:
q = (
    pl.scan_csv("data/larger.csv")
    .group_by("category")
    .agg(pl.col("invoice_total").sum())
)
result, timings = q.profile()
timings

node,start,end
str,u64,u64
"""optimization""",0,1
"""csv(data/larger.csv)""",1,18017
"""simple-projection(invoice_tota…",18018,18020
"""group_by_partitioned(category)""",18022,22608


### 11. See the optimizer at work: print `.explain(optimized=False)` and compare it to `.explain()`. The optimized plan pushes the projection/filter into the scan.

*ראו את ה-optimizer בפעולה: הדפיסו `.explain(optimized=False)` והשוו ל-`.explain()`. התוכנית המיטבית דוחפת את הבחירה/סינון לתוך הקריאה.*

In [11]:
q = (
    pl.scan_csv("data/larger.csv")
    .filter(pl.col("category") == "pens")
    .select("cust_num", "qty")
)
print("=== NOT optimized ===")
print(q.explain(optimized=False))
print("\n=== optimized ===")
print(q.explain())

=== NOT optimized ===
 SELECT [col("cust_num"), col("qty")] FROM
  FILTER [(col("category")) == (String(pens))] FROM
    Csv SCAN [data/larger.csv]
    PROJECT */10 COLUMNS

=== optimized ===
simple π 2/3 ["cust_num", "qty"]
  Csv SCAN [data/larger.csv]
  PROJECT 3/10 COLUMNS
  SELECTION: [(col("category")) == (String(pens))]


### 12. **Eager vs lazy timing.** Time (a) reading the whole file then filtering + grouping eagerly, versus (b) doing the same with `scan_csv` + `collect`. Print both durations.

*תזמון eager מול lazy. מדדו (א) קריאת כל הקובץ ואז סינון+קיבוץ, מול (ב) אותו דבר עם `scan_csv` + `collect`. הדפיסו את שני הזמנים.*

In [12]:
t0 = time.perf_counter()
eager = (pl.read_csv("data/larger.csv")
         .filter(pl.col("qty") > 5)
         .group_by("category").agg(pl.col("invoice_total").sum()))
eager_time = time.perf_counter() - t0

t0 = time.perf_counter()
lazy = (pl.scan_csv("data/larger.csv")
        .filter(pl.col("qty") > 5)
        .group_by("category").agg(pl.col("invoice_total").sum())
        .collect())
lazy_time = time.perf_counter() - t0

print(f"eager: {eager_time:.3f}s")
print(f"lazy : {lazy_time:.3f}s")

eager: 0.043s
lazy : 0.023s


### 13. Lazily parse `invoice_date_time` (string) into a real datetime and find the **earliest and latest** invoice timestamps.

*נתחו בעצלות את `invoice_date_time` לתאריך-שעה אמיתי ומצאו את החשבונית המוקדמת והמאוחרת ביותר.*

In [13]:
(
    pl.scan_csv("data/larger.csv")
    .select(
        first=pl.col("invoice_date_time").str.to_datetime().min(),
        last=pl.col("invoice_date_time").str.to_datetime().max(),
    )
    .collect()
)

first,last
datetime[μs],datetime[μs]
2019-01-01 00:00:01.581218,2019-12-31 23:59:48.992754


### 14. Build a **monthly revenue report** lazily: parse the timestamp, group by month, sum `invoice_total`, sort by month.

*בנו דוח הכנסות חודשי בעצלות: נתחו את החותמת, קבצו לפי חודש, סכמו `invoice_total`, מיינו לפי חודש.*

In [14]:
(
    pl.scan_csv("data/larger.csv")
    .group_by(
        month=pl.col("invoice_date_time").str.to_datetime().dt.month()
    )
    .agg(pl.col("invoice_total").sum().round(2).alias("revenue"))
    .sort("month")
    .collect()
)

month,revenue
i8,f64
1,7.2720e6
2,7.3252e6
3,7.3515e6
4,7.2790e6
5,7.3782e6
…,…
8,7.2641e6
9,7.2013e6
10,7.2465e6
